# Ingestão Bronze — Case

## Configuração do ambiente

Este bloco padroniza a nomenclatura e os caminhos do projeto, definindo o catálogo `case`, o schema `case.bronze` e o volume de entrada `/Volumes/case/bronze/inputs`. Em seguida, garante que esses três objetos existam no Unity Catalog (`CREATE IF NOT EXISTS`), deixando o ambiente pronto para a ingestão dos dados brutos.

In [0]:
# Padronização de nomenclatura e caminhos do projeto
catalog = "case"
bronze_schema = f"{catalog}.bronze"
landing_path = f"/Volumes/{catalog}/bronze/inputs"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.inputs")

print(f"Schema {bronze_schema} garantido no ambiente.")

## Ingestão dos CSVs para a camada Bronze

Este bloco percorre o dicionário `arquivos_tabelas`, que mapeia cada arquivo CSV do volume de entrada (`landing_path`) para a respectiva tabela Bronze (`tb_chamados`, `tb_chamados_hora`, `tb_clientes`, `tb_custos`, `tb_pesquisa_satisfacao`, `tb_canais`, `tb_base_motivos` e `tb_base_atendentes`).

Para cada arquivo encontrado no volume, ele:

* Lê o CSV com `inferSchema=True` (inferência de tipos para a primeira carga)
* Adiciona a coluna de auditoria `ingestion_datetime`, com o timestamp da ingestão
* Salva como tabela Delta no schema `case.bronze`, em modo `append`

Arquivos ausentes geram um alerta de erro indicando a necessidade de upload. Ao final, exibe uma amostra de 10 linhas de cada tabela ingerida para conferência.

In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento: "Nome_do_Arquivo.csv": "Nome_da_Tabela_Bronze"
arquivos_tabelas = {
    "chamados.csv": "tb_chamados",
    "Chamados_Hora.CSV": "tb_chamados_hora",
    "clientes.csv": "tb_clientes",
    "custos.csv": "tb_custos",
    "pesquisa_satisfacao.csv": "tb_pesquisa_satisfacao",
    "canais.csv": "tb_canais",
    "base_motivos.csv": "tb_base_motivos",
    "base_atendentes.csv": "tb_base_atendentes"
}

try:
    # Lista os arquivos fisicamente presentes no Volume
    arquivos_no_volume = [f.name for f in dbutils.fs.ls(landing_path)]
except Exception:
    arquivos_no_volume = []
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Verifique se o diretório existe.")

for arquivo, tabela in arquivos_tabelas.items():
    if arquivo in arquivos_no_volume:
        caminho = f"{landing_path}/{arquivo}"
        
        # Leitura bruta (inferSchema=True facilita a primeira carga)
        df_raw = spark.read.csv(caminho, header=True, inferSchema=True)
        
        # Adição da coluna de auditoria e salvamento
        df_raw \
            .withColumn("ingestion_datetime", current_timestamp()) \
            .write.format("delta").mode("append") \
            .saveAsTable(f"{bronze_schema}.{tabela}")

        print(f"[OK] {tabela} ingerida com sucesso ({df_raw.count()} linhas lidas do arquivo).")
    else:
        print(f"[ERRO] Arquivo '{arquivo}' não encontrado. Faça o upload para o Volume.")

# Debug
for tabela in arquivos_tabelas.values():
    print(f"\n--- {tabela} ---")
    display(spark.table(f"{bronze_schema}.{tabela}").limit(10))